코드 basic-cv

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np

model = LogisticRegression(max_iter=1000)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='f1_macro', n_jobs=-1)
print(f"폴드별 F1: {scores}")
print(f"평균: {scores.mean():.3f}, 표준편차: {scores.std():.3f}")

코드 bayesian-tuning

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'objective': 'binary',
        'learning_rate': trial.suggest_float('lr', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 256),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'verbose': -1
    }
    model = lgb.LGBMClassifier(**params, n_estimators=300)
    scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)
print(f"최적 F1: {study.best_value:.4f}")
print(f"최적 하이퍼파라미터: {study.best_params}")

코드 classification-eval

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# 1) 텍스트 형태의 분류 보고서
print(classification_report(y_test, y_pred,
      target_names=['진짜', '가짜'], digits=4))

# 2) 혼동 행렬
print("혼동 행렬:")
print(confusion_matrix(y_test, y_pred))

# 3) AUC와 AUPRC (양성 클래스 확률 사용)
auc = roc_auc_score(y_test, y_proba[:, 1])
ap = average_precision_score(y_test, y_proba[:, 1])
print(f"AUC: {auc:.4f}, AUPRC: {ap:.4f}")

# 4) 클래스별 F1 (다중 분류)
from sklearn.metrics import f1_score
print(f"매크로 F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"마이크로 F1: {f1_score(y_test, y_pred, average='micro'):.4f}")
print(f"가중 F1: {f1_score(y_test, y_pred, average='weighted'):.4f}")

코드 shap-explain

In [ ]:
import shap
import matplotlib.pyplot as plt

# TreeExplainer는 트리 기반 모델에 대해 정확한 SHAP 값을 빠르게 계산
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# 1) 전역 변수 중요도 (요약 그래프)
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                   plot_type='bar', show=False)
plt.tight_layout(); plt.show()

# 2) 변수별 효과 분포 (각 점 = 한 사례)
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                   show=False)
plt.tight_layout(); plt.show()

# 3) 한 개 사례의 국소 해석 (폭포 그래프)
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test[0],
        feature_names=feature_names
    ),
    show=False
)
plt.tight_layout(); plt.show()

코드 krippendorff

In [ ]:
import krippendorff
import numpy as np

# coder_data: shape = (코더 수, 사례 수), 결측치는 np.nan
# 예: 3명의 코더가 100개 댓글에 0/1/2 라벨을 부여
alpha = krippendorff.alpha(
    reliability_data=coder_data,
    level_of_measurement='nominal'    # 명목 척도
    # ordinal, interval, ratio도 가능
)
print(f"Krippendorff's α: {alpha:.4f}")

# 부트스트랩 신뢰구간 (1000회 재추출)
from sklearn.utils import resample
n = coder_data.shape[1]
alphas = []
for _ in range(1000):
    idx = resample(range(n), n_samples=n)
    alphas.append(krippendorff.alpha(
        reliability_data=coder_data[:, idx],
        level_of_measurement='nominal'
    ))
print(f"95% CI: [{np.percentile(alphas, 2.5):.4f}, "
      f"{np.percentile(alphas, 97.5):.4f}]")